# SupplyForge AI 2.0: KAN-25 - Data Cleaning & Preprocessing Pipeline



Tài liệu này thực hiện làm sạch dữ liệu giao dịch từ `order_1.csv` và danh mục sản phẩm từ `products.json`.

Dữ liệu làm sạch này sẽ được sử dụng để xây dựng chuỗi thời gian liên tục theo SKU, Feature Engineering, Label Engineering và huấn luyện mô hình dự báo nhu cầu.


### Mục tiêu & Quy tắc nghiệp vụ làm sạch:

1. **Cancelled & Returned Orders**:

   - Loại bỏ các đơn hàng có trạng thái bị hủy hoặc hoàn trả: `CANCELLED`, `TO_RETURN`, `IN_CANCEL`, `RETRY_SHIPPING`.

   - Chỉ giữ lại các đơn hàng hợp lệ: `COMPLETED`, `SHIPPED`.

2. **notCaculateMoney**:

   - Loại bỏ các dòng lỗi ghi nhận `notCaculateMoney == 1`.

3. **Duplicate Orders**:

   - Loại bỏ các đơn trùng lặp dựa trên khóa chính `_id`.

4. **Price & Cost Imputation**:

   - Điền khuyết các sản phẩm có giá bán (`price` <= 0) hoặc giá vốn (`cost` bị khuyết/NaN/<= 0) bằng giá trị trung vị (median) của nhóm danh mục `categoryId` tương ứng.

   - Nếu nhóm danh mục không có giá trị hoặc danh mục bị NaN, sử dụng giá trị trung vị toàn cầu (global median) làm fallback.

5. **Output**:

   - Lưu trữ dữ liệu đã làm sạch thành `order_1_cleaned.csv` và `products_cleaned.json` trong thư mục dữ liệu gốc.


In [1]:
import pandas as pd

import numpy as np

import json

import os



DATA_DIR = '/home/nguyen-dai-tung/supplyforge-ai/test-data/Database - NCKH'



print("Đang tải dữ liệu...")

df_orders = pd.read_csv(os.path.join(DATA_DIR, 'order_1.csv'), low_memory=False)

df_prod = pd.read_json(os.path.join(DATA_DIR, 'products.json'))



print(f"Tổng số đơn hàng ban đầu: {len(df_orders)}")

print(f"Tổng số sản phẩm ban đầu: {len(df_prod)}")


Đang tải dữ liệu...


Tổng số đơn hàng ban đầu: 49999
Tổng số sản phẩm ban đầu: 1335


In [2]:
# Kiểm toán nhanh chất lượng dữ liệu ban đầu

print("=== THỐNG KÊ TRƯỚC LÀM SẠCH ===")

print("1. Trạng thái đơn hàng (status):")

print(df_orders['status'].value_counts(dropna=False))



print("\n2. Cờ không tính tiền (notCaculateMoney):")

print(df_orders['notCaculateMoney'].value_counts(dropna=False))



print("\n3. Số lượng khóa chính _id bị trùng lặp ở orders:")

print(df_orders['_id'].duplicated().sum())



print("\n4. Sản phẩm có giá bán <= 0:")

print((df_prod['price'] <= 0).sum())



print("\n5. Sản phẩm có giá vốn (cost) bị khuyết (NaN/None):")

print(df_prod['cost'].isnull().sum())



print("\n6. Sản phẩm có giá vốn (cost) <= 0:")

print((df_prod['cost'] <= 0).sum())


=== THỐNG KÊ TRƯỚC LÀM SẠCH ===
1. Trạng thái đơn hàng (status):
status
COMPLETED    39369
CANCELLED    10630
Name: count, dtype: int64

2. Cờ không tính tiền (notCaculateMoney):
notCaculateMoney
0    49999
Name: count, dtype: int64

3. Số lượng khóa chính _id bị trùng lặp ở orders:
0

4. Sản phẩm có giá bán <= 0:
788

5. Sản phẩm có giá vốn (cost) bị khuyết (NaN/None):
1247

6. Sản phẩm có giá vốn (cost) <= 0:
0


In [ ]:
print("=== TIẾN HÀNH LÀM SẠCH ĐƠN HÀNG ===")
# 1. Lọc bỏ Cancelled & Returned
valid_statuses = ['COMPLETED', 'SHIPPED']
df_orders_cleaned = df_orders[df_orders['status'].isin(valid_statuses)].copy()
print(f"Sau khi lọc trạng thái (chỉ giữ {valid_statuses}): {len(df_orders_cleaned)} đơn")

# 2. Lọc bỏ notCaculateMoney == 1
df_orders_cleaned = df_orders_cleaned[df_orders_cleaned['notCaculateMoney'] != 1].copy()
print(f"Sau khi lọc notCaculateMoney != 1: {len(df_orders_cleaned)} đơn")

# 3. Lọc bỏ trùng lặp khóa chính _id
df_orders_cleaned.drop_duplicates(subset=['_id'], inplace=True)
print(f"Sau khi loại bỏ trùng lặp _id: {len(df_orders_cleaned)} đơn")

print("\nCác trạng thái còn lại trong đơn hàng:")
print(df_orders_cleaned['status'].value_counts(dropna=False))


=== TIẾN HÀNH LÀM SẠCH ĐƠN HÀNG ===
Sau khi lọc trạng thái (chỉ giữ ['COMPLETED', 'SHIPPED']): 39369 đơn
Sau khi lọc notCaculateMoney != 1: 39369 đơn
Sau khi loại bỏ trùng lặp _id: 39369 đơn

Các trạng thái còn lại trong đơn hàng:
status
COMPLETED    39369
Name: count, dtype: int64


In [4]:
print("=== TIẾN HÀNH ĐIỀN KHUYẾT GIÁ BÁN & GIÁ VỐN SẢN PHẨM ===")



# 1. Điền khuyết categoryId tạm thời

df_prod_cleaned = df_prod.copy()

df_prod_cleaned['cat_filled'] = df_prod_cleaned['categoryId'].fillna('unknown')



# 2. Tính trung vị toàn cầu làm fallback

global_median_price = df_prod_cleaned.loc[df_prod_cleaned['price'] > 0, 'price'].median()

global_median_cost = df_prod_cleaned.loc[df_prod_cleaned['cost'] > 0, 'cost'].median()

print(f"Trung vị giá bán toàn cầu: {global_median_price:,.2f}đ")

print(f"Trung vị giá vốn toàn cầu: {global_median_cost:,.2f}đ")



# 3. Tính trung vị theo nhóm danh mục

cat_median_prices = df_prod_cleaned[df_prod_cleaned['price'] > 0].groupby('cat_filled')['price'].median()

cat_median_costs = df_prod_cleaned[df_prod_cleaned['cost'] > 0].groupby('cat_filled')['cost'].median()



# 4. Áp dụng điền khuyết

def impute_row(row):

    price = row['price']

    cost = row['cost']

    cat = row['cat_filled']

    

    # Điền giá bán

    if price <= 0:

        imputed_price = cat_median_prices.get(cat, global_median_price)

        if pd.isna(imputed_price) or imputed_price <= 0:

            imputed_price = global_median_price

        price = imputed_price

        

    # Điền giá vốn

    if pd.isna(cost) or cost <= 0:

        imputed_cost = cat_median_costs.get(cat, global_median_cost)

        if pd.isna(imputed_cost) or imputed_cost <= 0:

            imputed_cost = global_median_cost

        cost = imputed_cost

        

    return pd.Series([price, cost])



df_prod_cleaned[['price', 'cost']] = df_prod_cleaned.apply(impute_row, axis=1)



# Xóa cột phụ trợ

df_prod_cleaned.drop(columns=['cat_filled'], inplace=True)



# 5. Kiểm tra kết quả sau điền khuyết

print("\nSố lượng sản phẩm giá bán <= 0 sau làm sạch:", (df_prod_cleaned['price'] <= 0).sum())

print("Số lượng sản phẩm giá vốn khuyết sau làm sạch:", df_prod_cleaned['cost'].isnull().sum())

print("Số lượng sản phẩm giá vốn <= 0 sau làm sạch:", (df_prod_cleaned['cost'] <= 0).sum())


=== TIẾN HÀNH ĐIỀN KHUYẾT GIÁ BÁN & GIÁ VỐN SẢN PHẨM ===
Trung vị giá bán toàn cầu: 250,000.00đ
Trung vị giá vốn toàn cầu: 18,500.00đ

Số lượng sản phẩm giá bán <= 0 sau làm sạch: 0
Số lượng sản phẩm giá vốn khuyết sau làm sạch: 0
Số lượng sản phẩm giá vốn <= 0 sau làm sạch: 0


In [5]:
print("=== LƯU TRỮ DỮ LIỆU ĐÃ LÀM SẠCH ===")



order_cleaned_path = os.path.join(DATA_DIR, 'order_1_cleaned.csv')

product_cleaned_path = os.path.join(DATA_DIR, 'products_cleaned.json')



# Lưu orders dạng CSV

df_orders_cleaned.to_csv(order_cleaned_path, index=False)

print(f"Đã lưu orders sạch tại: {order_cleaned_path}")



# Lưu products dạng JSON có format thụt lề

prod_records = df_prod_cleaned.to_dict(orient='records')

with open(product_cleaned_path, 'w', encoding='utf-8') as f:

    json.dump(prod_records, f, indent=2, ensure_ascii=False)

print(f"Đã lưu products sạch tại: {product_cleaned_path}")


=== LƯU TRỮ DỮ LIỆU ĐÃ LÀM SẠCH ===


Đã lưu orders sạch tại: /home/nguyen-dai-tung/supplyforge-ai/test-data/Database - NCKH/order_1_cleaned.csv
Đã lưu products sạch tại: /home/nguyen-dai-tung/supplyforge-ai/test-data/Database - NCKH/products_cleaned.json


In [6]:
# Load lại và kiểm chứng chất lượng file đầu ra

print("=== BÁO CÁO ĐỊNH LƯỢNG SAU KHI LÀM SẠCH VÀ LƯU FILE ===")

df_orders_check = pd.read_csv(order_cleaned_path, low_memory=False)

df_prod_check = pd.read_json(product_cleaned_path)



print(f"1. Tổng số dòng trong file orders sạch: {len(df_orders_check)}")

print(f"   Tỷ lệ giảm: {(len(df_orders) - len(df_orders_check)) / len(df_orders) * 100:.2f}%")

print(f"2. Trạng thái trong file orders sạch: {df_orders_check['status'].unique().tolist()}")

print(f"3. Cờ notCaculateMoney trong file orders sạch: {df_orders_check['notCaculateMoney'].unique().tolist()}")

print(f"4. Số trùng lặp _id trong file orders sạch: {df_orders_check['_id'].duplicated().sum()}")

print(f"5. Sản phẩm giá bán <= 0 trong file products sạch: {(df_prod_check['price'] <= 0).sum()}")

print(f"6. Sản phẩm giá vốn khuyết trong file products sạch: {df_prod_check['cost'].isnull().sum()}")


=== BÁO CÁO ĐỊNH LƯỢNG SAU KHI LÀM SẠCH VÀ LƯU FILE ===


1. Tổng số dòng trong file orders sạch: 39369
   Tỷ lệ giảm: 21.26%
2. Trạng thái trong file orders sạch: ['COMPLETED']
3. Cờ notCaculateMoney trong file orders sạch: [0]
4. Số trùng lặp _id trong file orders sạch: 0
5. Sản phẩm giá bán <= 0 trong file products sạch: 0
6. Sản phẩm giá vốn khuyết trong file products sạch: 0
